In [ ]:
import pandas as pd
df = pd.read_csv("admission_data.csv")
df.info()
df.isnull().sum()
df.columns = df.columns.str.strip()
X = df.drop("Chance of Admit", axis=1)
y = df["Chance of Admit"]
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   GRE Score          500 non-null    int64  
 1   TOEFL Score        500 non-null    int64  
 2   University Rating  500 non-null    int64  
 3   SOP                500 non-null    float64
 4   LOR                500 non-null    float64
 5   CGPA               500 non-null    float64
 6   Research           500 non-null    int64  
 7   Chance of Admit    500 non-null    float64
dtypes: float64(4), int64(4)
memory usage: 31.4 KB


In [ ]:
!pip install -q catboost xgboost

from sklearn.linear_model import Ridge, Lasso

model = Ridge(alpha=1.0)  # Try Lasso(alpha=0.1) too
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("R2 Score:", r2_score(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))

In [ ]:
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import r2_score, mean_squared_error
regressors = [
    ('lr', LinearRegression()),
    ('rf', RandomForestRegressor(n_estimators=100, random_state=42)),
    ('dt', DecisionTreeRegressor(random_state=42)),
    ('mlp', MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=500, random_state=42)),
    ('knn', KNeighborsRegressor(n_neighbors=5)),
    ('svr', SVR(kernel='rbf')),
    ('xgb', XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=4, random_state=42, verbosity=0)),
    ('cat', CatBoostRegressor(iterations=100, learning_rate=0.1, depth=4, verbose=0, random_state=42))
]
meta_learner = LinearRegression()
stacking_model = StackingRegressor(
    estimators=regressors,
    final_estimator=meta_learner,
    passthrough=True,  # includes original features in meta-learner input
    n_jobs=-1
)
stacking_model.fit(X_train, y_train)
y_pred_stack = stacking_model.predict(X_test)
print("R2 Score:", r2_score(y_test, y_pred_stack))
print("MSE:", mean_squared_error(y_test, y_pred_stack))

R2 Score: 0.8254053945203386
MSE: 0.0035704596820590757




# Save the model
joblib.dump(model, "admission_model.pkl")

# Save the scaler as well (important for future data preprocessing)
joblib.dump(scaler, "scaler.pkl")

In [ ]:
import joblib
joblib.dump(stacking_model, "admission_model.pkl")
joblib.dump(scaler, "scaler.pkl")
joblib.dump(X_train, "X_train.pkl")
print("✅ Model and scaler saved successfully.")

✅ Model and scaler saved successfully.


In [ ]:
import joblib
import numpy as np
loaded_model = joblib.load("admission_model.pkl")
loaded_scaler = joblib.load("scaler.pkl")
sample = np.array([[315, 110, 4, 4, 4.5, 9.2, 1]])
sample_scaled = loaded_scaler.transform(sample)
pred = loaded_model.predict(sample_scaled)
print("Predicted Chance of Admission: {:.2f}%".format(pred[0] * 100))

Predicted Chance of Admission: 82.93%


/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
